# 05 — Workflows: when fixed control flow beats an agent

**What you'll learn**

- Four of the five workflow patterns in Anthropic's *Building Effective AI Agents* — routing, prompt-chaining, parallelization, orchestrator-workers (evaluator-optimizer, the fifth, is chapter 06) — and the table that says when each beats a model in a loop
- Build `route(ticket)` — one cheap classifier call that sorts each ticket into `simple_refund`, `needs_policy`, or `escalate_review`, so easy tickets never pay for the full agent
- Chain `extract` -> gate -> `rules.decide` into deterministic control flow, where a failed extraction stops the chain instead of feeding garbage forward
- Fan `run_agent` out over the dev split with a throttled `ThreadPoolExecutor` and read the speedup off `LEDGER` call latencies
- Sketch an orchestrator that splits a batch by lane and dispatches workers, then read the decision table for when fixed flow wins over an agent

*Time: ~6 min on a first live run (the parallel fan-out is ~80 s of it); ~1 min cached. Cost: ~$0.02. Cached reruns are free.*

## The cheapest agent is the one you didn't write

Chapter 02 gave the model the wheel: a loop that decides its own next tool call, every step. That is the right tool when the path genuinely depends on what the model finds — but most ops-desk tickets do not. A ticket past the return window is a deny; a flagged serial returner is an escalate; a clean in-window refund is arithmetic. Handing those to a self-directing agent pays the full price of a tool loop — several model calls, several seconds — to reach a conclusion a single branch would have reached.

A *workflow* is the alternative: code you wrote, with model calls at the points you chose and plain control flow everywhere else. Anthropic's [*Building Effective AI Agents*](https://www.anthropic.com/engineering/building-effective-agents) draws the line precisely — workflows are systems where LLMs and tools are orchestrated through predefined code paths; agents are systems where the model directs its own tool use — and catalogs five workflow shapes worth knowing by name. This chapter builds four of them against the same tickets — the fifth, evaluator-optimizer, is chapter 06's subject — then closes with the table that tells you which to reach for.

| Pattern | Shape | Wins when |
|---|---|---|
| Routing | classify, then dispatch to a handler | inputs fall into distinct kinds that want different handling |
| Prompt-chaining | fixed ordered steps, a gate between them | the task decomposes into subtasks you can check at the seam |
| Parallelization | fan the same step over many inputs | inputs are independent and the step is dominated by waiting |
| Orchestrator-workers | a lead splits the work, workers run it | the split is dynamic but each piece is routine |

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

Every model call below still goes through `shoplab.llm.complete`, so Phoenix traces the workflow calls exactly as it traced the agent — and a routed ticket makes a visibly shorter trace than a full loop. Optional as ever: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Routing: one cheap call picks the lane

Routing spends one small model call up front to answer a single question — what kind of ticket is this? — and then lets ordinary code send each kind to a handler built for it. The classifier returns a label from a fixed set, nothing else; the dispatch is a dictionary of branches, not a model. The win is that the easy lanes never pay for the hard lane's machinery.

Three lanes: `simple_refund` for a clear in-window refund, `needs_policy` when the outcome hinges on a specific policy, and `escalate_review` for fraud signals. The prompt lists the lanes, asks for the lane name alone, and the parser keeps only a known label — anything unrecognized falls back to `needs_policy`, the safe default that does the most work.

In [ ]:
import shoplab.llm
from shoplab.world import load_tickets, load_orders, load_customers

tickets = load_tickets()
orders = {o["order_id"]: o for o in load_orders()}
customers = {c["customer_id"]: c for c in load_customers()}

LANES = ("simple_refund", "needs_policy", "escalate_review")

def route(ticket):
    prompt = ("Classify this return ticket into exactly one handling lane.\n"
              "simple_refund: a clear in-window refund or return, no judgment call.\n"
              "needs_policy: the outcome hinges on a policy (damage, defect, hazmat, "
              "international, restocking fee).\n"
              "escalate_review: fraud signals, a flagged serial returner, or a large "
              "unevidenced claim.\n"
              "Answer with only the lane name.\n\nCustomer message: " + ticket["reason_text"])
    out = shoplab.llm.llm(prompt, model=MODEL).strip().lower()
    return next((lane for lane in LANES if lane in out), "needs_policy")

for t in tickets["dev"][:4]:
    print(f"{t['ticket_id']}  ->  {route(t)}")

> **What you should see:** four tickets sorted into lanes with one model call each — on our frozen run two land in `escalate_review` and two in `simple_refund`. None fell into `needs_policy` in this handful, which is fine: that lane, and the full agent behind it, shows up in the dispatch below. The labels need not match the gold *decision*; routing is triage, not adjudication. What matters is that each lane is one cheap call to compute and maps to a handler of the right weight.

## Steering to the right-sized handler

The lane label only pays off if the handlers differ in cost, so they do. `escalate_review` returns immediately with zero model calls — a flagged ticket goes to a human, no deliberation. `simple_refund` takes one shot: a single call that reads the ticket and returns the decision JSON, no tool loop. `needs_policy` gets the full chapter-02 agent, tools and all, because those are the tickets that actually need to look things up. Same nine-tool desk on offer; the router decides who is allowed to spend it.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import standard_tools

SYSTEM05 = ("You are the Larkspur Outfitters returns desk. Look up the order, the "
            "customer, and the one policy that applies, then decide. Do not look "
            "anything up twice. Call finish as soon as you are sure, giving decision, "
            "policy_id, and refund_usd (number or null).")

def render_ticket(t):
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']}, qty {t['qty']}, condition "
            f"{t['item_condition']}, days since delivery {t['days_since_delivery']}, "
            f"photo evidence {t['evidence_photo']}, requested action "
            f"{t['requested_action']}. Customer writes: {t['reason_text']}")

def run_ticket05(t):
    return run_agent(render_ticket(t), standard_tools(),
                     system=SYSTEM05, max_steps=8).answer

In [ ]:
def one_shot(t):
    p = (render_ticket(t) + '\nReply with ONLY JSON {"decision":_,"policy_id":_,'
         '"refund_usd":_}; decision in approve_refund|partial_refund|replacement|'
         "store_credit|deny|escalate.")
    try:
        return shoplab.llm.parse_json_loose(shoplab.llm.llm(p, model=MODEL))
    except ValueError:
        return {"decision": None}

def handle(t, lane):
    if lane == "escalate_review":
        return {"decision": "escalate"}
    return one_shot(t) if lane == "simple_refund" else (run_ticket05(t) or {})

In [ ]:
lane_of = {t["ticket_id"]: route(t) for t in tickets["dev"]}
demo = [next(t for t in tickets["dev"] if lane_of[t["ticket_id"]] == lane) for lane in LANES]
for t in demo:
    lane = lane_of[t["ticket_id"]]
    print(f"{t['ticket_id']}  {lane:<16}  routed -> {str(handle(t, lane).get('decision')):<14}"
          f"  gold {t['gold']['decision']}")

In [ ]:
from shoplab.llm import LEDGER

def calls(fn, items):
    n = len(LEDGER)
    for t in items:
        fn(t)
    return len(LEDGER) - n

sample = tickets["dev"][:4]
print("routed model calls:    ", calls(lambda t: handle(t, route(t)), sample))
print("full-agent model calls:", calls(run_ticket05, sample))

> **What you should see:** the dispatch handles one ticket per lane and, on our frozen run, gets all three right — the flagged ticket escalates with no model call, the clean refund resolves in a single `one_shot`, and only the policy-dependent ticket spends a full agent loop. Then the count makes the economics blunt: the routed pipeline spends a handful of calls over the four-ticket sample where running the full agent on all four costs several times more, because most tickets never enter the loop. That is the whole case for routing — you pay the loop's price only where it earns it. The trade shows in the cheap lane: `one_shot` sometimes invents a policy id or a non-numeric amount, so route the risky cases to the agent, not away from it.

## Prompt-chaining: extract, gate, decide

Chaining is the workflow you reach for when a task splits into ordered steps and you can check the handoff between them. Here the task is two steps: pull the structured fields out of the customer's prose, then apply the rules. The model does step one — reading `damaged` and `refund` out of a paragraph is exactly what it is good at. Step two is `shoplab.rules.decide`, pure code, no model at all.

Between the steps sits a *gate*: a plain predicate that checks the extraction is well-formed before the rules ever see it. If the model returns junk — a made-up condition, a missing field, unparseable text — the gate stops the chain and escalates rather than feeding garbage into a decision. That gate is the whole reason to write this as a chain instead of one big prompt: the control flow is yours, and a bad model step cannot route around it.

In [ ]:
from shoplab.rules import decide

CONDS = {"unopened", "opened", "damaged", "defective"}
ACTS = {"refund", "replacement", "store_credit"}

def extract(reason_text):
    p = ('Extract JSON with keys "item_condition" (unopened|opened|damaged|'
         'defective) and "requested_action" (refund|replacement|store_credit) '
         "from this message. Return only JSON.\n\nMessage: " + reason_text)
    return shoplab.llm.parse_json_loose(shoplab.llm.llm(p, model=MODEL))

def gate_ok(f):
    return (isinstance(f, dict) and f.get("item_condition") in CONDS
            and f.get("requested_action") in ACTS)

In [ ]:
def chain(t):
    order, customer = orders[t["order_id"]], customers[t["customer_id"]]
    try:
        fields = extract(t["reason_text"])
    except ValueError:
        return {"decision": "escalate", "stopped": "extraction not JSON"}
    if not gate_ok(fields):
        return {"decision": "escalate", "stopped": "gate rejected fields"}
    return decide({**t, **fields}, order, customer)

for t in tickets["dev"][:2]:
    print(t["ticket_id"], "chain ->", chain(t))
    print(" " * 9, "gold ->", t["gold"])
print("gate on good fields:", gate_ok({"item_condition": "opened", "requested_action": "refund"}))
print("gate on junk:       ", gate_ok({"item_condition": "maybe", "requested_action": None}))

> **What you should see:** for a clearly-worded ticket the chain reproduces the gold decision — `extract` recovers the same `item_condition` and `requested_action` the ticket already carries, and `decide` does the rest deterministically (note the serial-returner ticket still escalates, because step 1 of the cascade fires before any extracted field matters). The gate returns `True` on well-formed fields and `False` on the junk dict: `decide` is never handed fields it would choke on. What you are watching is control flow you own — one model call wedged between a parser and a rules engine, not a model steering the whole task.

## Parallelization: fan the loop out, throttle the fan

The sections so far each spent model calls more cleverly. Parallelization spends them at the same time. The dev split is twelve independent tickets; running the agent on them one after another adds twelve latencies end to end, and almost all of that time is the network waiting on the model, not your CPU working. That is the textbook case for concurrency — overlap the waits.

`ThreadPoolExecutor` is enough: the work is I/O-bound, so threads (not processes) are right, and the GIL never blocks a socket waiting on a response. `max_workers` is the throttle, and it is small on purpose — this box has four cores, the provider has rate limits, and a fan you cannot bound is a fan that trips them. Four in flight turns a serial wait into a parallel one without picking a fight with either limit.

In [ ]:
import time
import concurrent.futures as cf

def parallel_map(fn, items, max_workers=4):
    with cf.ThreadPoolExecutor(max_workers=max_workers) as ex:
        return list(ex.map(fn, items))

dev = tickets["dev"]
start = len(LEDGER)
t0 = time.perf_counter()
preds = parallel_map(run_ticket05, dev, max_workers=4)
wall = time.perf_counter() - t0
serial_equiv = sum(row["seconds"] for row in LEDGER[start:])
print(f"{len(dev)} tickets, {len(LEDGER) - start} model calls, 4 workers")
print(f"parallel wall time:      {wall:6.1f} s")
print(f"serial-equivalent (sum): {serial_equiv:6.1f} s")
print(f"speedup: {serial_equiv / wall:.1f}x   throughput: {len(dev) / wall:.2f} tickets/s")

> **What you should see:** the sum of the individual call latencies runs several times longer than the wall-clock the fan-out actually took — expect a speedup approaching, but under, the worker count, since each ticket's own calls are still serial inside its thread. The serial-equivalent line is what a plain `for` loop would pay: it makes exactly these calls, back to back. Rerun the cell and the absolute times collapse — litellm's disk cache answers every call in milliseconds — but the speedup does not: even a cached call carries real per-call latency, and four workers still overlap it, so the ratio stays near the worker count. That the win survives caching is the tell — the work was always wait-bound, not CPU-bound, whether the wait is a network round-trip or a cache read.

## Orchestrator-workers: a lead splits, workers run

The last pattern adds one move on top of routing and parallelization: a lead step that reads the whole batch and decides how to break it up, before any worker runs. Anthropic's Research feature is built this way — [a lead agent that coordinates the process while delegating to specialized subagents that operate in parallel](https://www.anthropic.com/engineering/multi-agent-research-system). Our version is deliberately small: one orchestrator call writes the batch's headline, `route` fans out to label every ticket, `handle` fans out again to clear them, and the plan comes back merged.

Nothing here is new machinery — it is `route`, `parallel_map`, and `handle`, composed. That is the pattern's point: the lead owns the split, the workers own the toil, and the code owns the flow.

In [ ]:
def orchestrate(batch):
    headline = shoplab.llm.llm(
        "You are the ops lead. Today's return tickets:\n"
        + "\n".join(f"- {t['ticket_id']}: {t['reason_text'][:70]}" for t in batch)
        + "\nIn one sentence, name the batch's main risk.", model=MODEL)
    lanes = parallel_map(route, batch, max_workers=4)                     # workers split
    preds = parallel_map(lambda tl: handle(*tl), list(zip(batch, lanes)),
                         max_workers=4)                                   # workers run
    plan = {}
    for t, lane in zip(batch, lanes):
        plan.setdefault(lane, []).append(t["ticket_id"])
    return {"headline": headline.strip(), "plan": plan, "preds": preds}

result = orchestrate(tickets["dev"][:6])
print("headline:", result["headline"])
for lane, ids in result["plan"].items():
    print(f"  {lane:<16} {ids}")

> **What you should see:** one headline sentence from the lead call, then the six tickets grouped by lane — the plan the workers executed. Both fan-outs reuse the cached labels and decisions from the sections above, so the orchestrator adds a plan and a summary, not a rerun. Scale the batch and only the lead call stays serial; the workers were parallel the moment they went through `parallel_map`.

## When fixed flow wins

An agent earns its overhead when the path is genuinely unknown until the model sees the data — open-ended research, multi-step debugging, anything where the next tool depends on the last result. A workflow wins everywhere the shape is known in advance, which is more of production than the hype admits. The tickets are the proof: routing clears most of them without ever entering a loop.

| Reach for | When |
|---|---|
| Fixed workflow | the steps are known ahead of time; you want predictable cost and latency and a place to put a gate |
| Routing | inputs split into kinds that deserve different handling, and misroutes are cheap to recover from |
| Parallelization | the inputs are independent and the work is dominated by waiting |
| A full agent | the path truly depends on what the model finds, and the task is worth several calls to explore |

The honest default is to start with the workflow and add agency only where a fixed path visibly fails — cheaper to run, easier to trace, and far easier to test than a model that redecides its plan every step.

## Recap

| Concept | One-liner |
|---|---|
| Workflow vs agent | a workflow is code you wrote with model calls at chosen points; an agent directs its own tool use. |
| Routing | one cheap classifier call picks a lane; a dict of handlers dispatches, and easy lanes skip the loop. |
| Prompt-chaining | ordered steps with a gate at the seam; a failed `extract` stops the chain before `decide` runs. |
| Gate | a plain predicate the model cannot route around — the reason to chain instead of one big prompt. |
| Parallelization | `ThreadPoolExecutor` overlaps I/O-bound calls; `max_workers` throttles for cores and rate limits. |
| Serial-equivalent | sum of per-call `LEDGER` latencies — what a serial loop pays, and the honest speedup baseline. |
| Orchestrator-workers | a lead call splits the batch; `route` and `handle` fan out as workers; code merges the result. |
| The default | start with the workflow; add agency only where a fixed path visibly fails. |

## Exercises

1. Add a fourth lane. Insert `needs_math` for tickets whose refund is a non-trivial calculation — opened items with a restocking fee, damaged items with shipping added back — route to it, and give it a handler that runs `one_shot` then rechecks the amount with `shoplab.tools.calc`. Does pulling the arithmetic-heavy tickets out of `needs_policy` change how often the routed pipeline matches gold?
2. Make the chain recover instead of escalate. When `gate_ok` rejects the extraction, retry `extract` once with a stricter prompt that names the exact allowed values, and only escalate if the second attempt also fails. On the dev split, how many gate rejections does the retry rescue, and at what cost in extra calls?
3. Measure the speedup curve. Run the parallelization cell at `max_workers` 1, 2, and 4 — give each run a fresh `SYSTEM05` string so it starts cold — and tabulate wall time against workers. Where does the curve stop paying off, and does the knee line up with the four cores, the provider's rate limit, or the per-ticket call chain that stays serial?

**Next up:** chapter 06 stops trusting the decision and starts checking it — an LLM-as-judge that grades the agent's own answers, and the evaluator-optimizer loop that refines until they pass.